# Graph Visualization with Cosmograph

This notebook visualizes the semantic network graph created from daily notes using Cosmograph's GPU-accelerated force layout visualization.

## Import Libraries

In [1]:
import pandas as pd
from cosmograph import cosmo
from pathlib import Path

## Load Data

Load the nodes and edges CSV files generated by the semantic network.

In [2]:
# Load nodes and edges
nodes = pd.read_csv('nodes.csv')
edges = pd.read_csv('edges.csv')

print(f"Loaded {len(nodes)} nodes and {len(edges)} edges")
print(f"\nNodes columns: {nodes.columns.tolist()}")
print(f"Edges columns: {edges.columns.tolist()}")

# Preview the data
print("\nFirst few nodes:")
print(nodes.head(3))
print("\nFirst few edges:")
print(edges.head(3))

Loaded 5109 nodes and 1430 edges

Nodes columns: ['node_id', 'label']
Edges columns: ['source', 'target', 'weight']

First few nodes:
   node_id                                              label
0        0  # 01-07-2024\n\nIn the beginning of the mornin...
1        1  After that i did some standard security traini...
2        2  \nAfter this Thijs discussed the current data ...

First few edges:
   source  target  weight
0       5     380     1.0
1       5     476     1.0
2       5     538     1.0


## Prepare Data for Visualization

Create shorter labels and add additional metadata for better visualization.

In [3]:
# Create a short label for display (first 50 chars)
nodes['short_label'] = nodes['label'].str[:50] + '...'

# Calculate node degree (number of connections) for sizing
from collections import Counter
node_degree = Counter()
for _, edge in edges.iterrows():
    node_degree[edge['source']] += 1
    node_degree[edge['target']] += 1

nodes['degree'] = nodes['node_id'].map(node_degree).fillna(0)

# Add a category based on degree (highly connected vs isolated)
nodes['category'] = pd.cut(
    nodes['degree'], 
    bins=[0, 1, 5, 10, float('inf')],
    labels=['Isolated', 'Low', 'Medium', 'High']
)

print(f"Node degree statistics:")
print(nodes['degree'].describe())
print(f"\nCategory distribution:")
print(nodes['category'].value_counts())

Node degree statistics:
count    5109.000000
mean        0.559796
std         2.621405
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        28.000000
Name: degree, dtype: float64

Category distribution:
category
Isolated    556
Low         258
High         68
Medium       21
Name: count, dtype: int64


## Visualization 1: Basic Graph with Force Layout

Simple visualization showing all nodes and edges with force-directed layout.

In [4]:
basic_graph = cosmo(
    points=nodes,
    links=edges,
    point_id_by='node_id',
    link_source_by='source',
    link_target_by='target',
    point_label_by='short_label',
    point_size_by='degree',
    point_size_scale=2,
    disable_point_size_legend=True
)
basic_graph

## Visualization 2: Color by Connectivity Category

Color nodes based on their connectivity level (isolated, low, medium, high).

In [5]:
colored_graph = cosmo(
    points=nodes,
    links=edges,
    point_id_by='node_id',
    link_source_by='source',
    link_target_by='target',
    point_label_by='short_label',
    point_color_by='category',
    point_size_by='degree',
    point_size_scale=2,
    disable_point_size_legend=True
)
colored_graph

## Visualization 3: With Edge Weights

Visualize edge weights (similarity scores) to see stronger connections.

In [6]:
weighted_graph = cosmo(
    points=nodes,
    links=edges,
    point_id_by='node_id',
    link_source_by='source',
    link_target_by='target',
    link_width_by='weight',
    point_label_by='short_label',
    point_color_by='category',
    point_size_by='degree',
    point_size_scale=2,
    disable_point_size_legend=True
)
weighted_graph

## Visualization 4: Filtered - Only Strong Connections

Filter to show only edges with weight > 0.5 to focus on strongly connected notes.

In [7]:
# Filter edges by weight
strong_edges = edges[edges['weight'] > 0.5].copy()

# Get only nodes that are in the filtered edges
connected_node_ids = set(strong_edges['source']).union(set(strong_edges['target']))
filtered_nodes = nodes[nodes['node_id'].isin(connected_node_ids)].copy()

print(f"Filtered to {len(filtered_nodes)} nodes and {len(strong_edges)} edges")

filtered_graph = cosmo(
    points=filtered_nodes,
    links=strong_edges,
    point_id_by='node_id',
    link_source_by='source',
    link_target_by='target',
    link_width_by='weight',
    point_label_by='short_label',
    point_color_by='category',
    point_size_by='degree',
    point_size_scale=3,
    disable_point_size_legend=True
)
filtered_graph

Filtered to 136 nodes and 724 edges


## Interactive Exploration

You can interact with the graphs above and use the widget object to explore further.

In [8]:
# Recenter the view if you've lost your graph
# weighted_graph.fit_view()

# Get selected point IDs after selecting nodes in the visualization
# selected_ids = weighted_graph.selected_point_ids
# print(f"Selected nodes: {selected_ids}")

# You can then filter the nodes dataframe to see details
# if selected_ids:
#     selected_nodes = nodes[nodes['node_id'].isin(selected_ids)]
#     print(selected_nodes[['node_id', 'label', 'degree']])

## Graph Statistics

Explore some basic statistics about the network structure.

In [9]:
print(f"Total nodes: {len(nodes)}")
print(f"Total edges: {len(edges)}")
print(f"\nEdge weight statistics:")
print(edges['weight'].describe())

print(f"\nTop 10 most connected nodes:")
top_nodes = nodes.nlargest(10, 'degree')[['node_id', 'short_label', 'degree']]
for idx, row in top_nodes.iterrows():
    print(f"  Node {row['node_id']} ({row['degree']} connections): {row['short_label']}")

Total nodes: 5109
Total edges: 1430

Edge weight statistics:
count    1430.000000
mean        0.667261
std         0.319928
min         0.300129
25%         0.341484
50%         0.527188
75%         1.000000
max         1.000000
Name: weight, dtype: float64

Top 10 most connected nodes:
  Node 380 (28 connections): 
...
  Node 476 (28 connections): 
...
  Node 538 (28 connections): 
...
  Node 545 (28 connections): 
...
  Node 559 (28 connections): 
...
  Node 584 (28 connections): 
...
  Node 626 (28 connections): 
...
  Node 704 (28 connections): 
...
  Node 724 (28 connections):  ...
  Node 939 (28 connections): 
...
